## Importy


In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import sklearn
import m2cgen as m2c

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Diagnostyka środowiska


In [2]:
print("Python:", sys.version)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.10.10 (tags/v3.10.10:aad5f6a, Feb  7 2023, 17:20:36) [MSC v.1929 64 bit (AMD64)]
NumPy: 2.2.6
pandas: 2.3.3
scikit-learn: 1.7.2


## Wczytanie danych


In [3]:
df = pd.read_csv("../data/Student_Performance.csv")
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


## Struktura danych, podstawowe statystyki i sprawdzenie braków danych


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  object 
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 468.9+ KB


In [5]:
df.describe()

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


In [6]:
df.isnull().sum()

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

## Przygotowanie kolumn


In [7]:
df = df.rename(columns={
    "Extracurricular Activities": "Extra Activities"
})

Ustawiam prostszą nazwę dla kolumny.


## Zmiana wartości Yes/No na 1/0


In [8]:
df["Extra Activities"] = (
    df["Extra Activities"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0
    })
)

## Wybór cech i targetu


In [9]:
feature_columns = [
    "Hours Studied",
    "Previous Scores",
    "Extra Activities",
    "Sleep Hours",
    "Sample Question Papers Practiced"
]

X = df[feature_columns].copy()
y = df["Performance Index"].copy()

## Czyszczenie danych przed treningiem


In [10]:
for col in feature_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y = pd.to_numeric(y, errors="coerce")

data = X.copy()
data["Performance Index"] = y

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna()

X = data[feature_columns].astype("float64")
y = data["Performance Index"].astype("float64")

## Diagnostyka danych po czyszczeniu


In [11]:
print("X dtypes:")
print(X.dtypes)

print("\nBraki w X:")
print(X.isnull().sum())

print("\nInf w X:")
print(np.isinf(X).sum())

print("\ny dtype:")
print(y.dtype)

print("\nBraki w y:")
print(y.isnull().sum())

print("\nInf w y:")
print(np.isinf(y).sum())

print("\ny describe:")
print(y.describe())

X dtypes:
Hours Studied                       float64
Previous Scores                     float64
Extra Activities                    float64
Sleep Hours                         float64
Sample Question Papers Practiced    float64
dtype: object

Braki w X:
Hours Studied                       0
Previous Scores                     0
Extra Activities                    0
Sleep Hours                         0
Sample Question Papers Practiced    0
dtype: int64

Inf w X:
Hours Studied                       0
Previous Scores                     0
Extra Activities                    0
Sleep Hours                         0
Sample Question Papers Practiced    0
dtype: int64

y dtype:
float64

Braki w y:
0

Inf w y:
0

y describe:
count    10000.000000
mean        55.224800
std         19.212558
min         10.000000
25%         40.000000
50%         55.000000
75%         71.000000
max        100.000000
Name: Performance Index, dtype: float64


## Podział na zbiór treningowy i testowy


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Sprawdzenie rozmiarów zbiorów


In [13]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 5)
X_test: (2000, 5)
y_train: (8000,)
y_test: (2000,)


## Trening modelu


In [14]:
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## Diagnostyka modelu po treningu


In [15]:
print("coef:")
print(model.coef_)

print("\nintercept:")
print(model.intercept_)

print("\nInf w coef:")
print(np.isinf(model.coef_).sum())

print("\nNaN w coef:")
print(np.isnan(model.coef_).sum())

print("\nCzy intercept jest inf?")
print(np.isinf(model.intercept_))

print("\nCzy intercept jest nan?")
print(np.isnan(model.intercept_))

coef:
[2.85248393 1.0169882  0.60861668 0.47694148 0.19183144]

intercept:
-33.921946215556126

Inf w coef:
0

NaN w coef:
0

Czy intercept jest inf?
False

Czy intercept jest nan?
False


## Predykcja na zbiorze testowym


Predykcję wykonałem jawnie ze wzoru regresji liniowej. W moim środowisku `model.predict(X_test)` zwracał poprawne wyniki, ale generował ostrzeżenia numeryczne z operacji `matmul`. Jawne użycie `dot()` wykonuje tę samą operację dla regresji liniowej: mnożę cechy przez współczynniki modelu, a następnie dodaję wyraz wolny.

In [16]:
X_test_np = X_test.to_numpy(dtype=np.float64)
y_pred = X_test_np.dot(model.coef_) + model.intercept_

print(y_pred[:10])
print("Inf w y_pred:", np.isinf(y_pred).sum())
print("NaN w y_pred:", np.isnan(y_pred).sum())

[54.71185392 22.61551294 47.90314471 31.28976748 43.00457042 59.07125171
 45.90347516 86.45911791 37.70014037 72.05592481]
Inf w y_pred: 0
NaN w y_pred: 0


## Ocena jakości modelu


In [17]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 1.6111213463123044
MSE: 4.082628398521853
RMSE: 2.0205515085050054
R2: 0.9889832909573145


## Zapis metryk do pliku


In [18]:
metrics = {
    "MAE": float(mae),
    "MSE": float(mse),
    "RMSE": float(rmse),
    "R2": float(r2)
}

os.makedirs("../exports", exist_ok=True)

with open("../exports/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

metrics

{'MAE': 1.6111213463123044,
 'MSE': 4.082628398521853,
 'RMSE': 2.0205515085050054,
 'R2': 0.9889832909573145}

## Eksport modelu do JavaScript


In [19]:
js_code = m2c.export_to_javascript(model)

os.makedirs("../exports", exist_ok=True)

with open("../exports/model.js", "w") as f:
    f.write(js_code)

print("Model zapisany do ../exports/model.js")

Model zapisany do ../exports/model.js


## Przykładowa predykcja dla jednego rekordu


In [20]:
sample = X_test.iloc[[0]]
sample_np = sample.to_numpy(dtype=np.float64)

sample_prediction = sample_np.dot(model.coef_) + model.intercept_

print("Dane wejściowe:")
print(sample)

print("\nPredykcja Performance Index:")
print(float(sample_prediction[0]))

print("\nRzeczywista wartość Performance Index:")
print(float(y_test.iloc[0]))

Dane wejściowe:
      Hours Studied  Previous Scores  Extra Activities  Sleep Hours  \
6252            5.0             69.0               0.0          8.0   

      Sample Question Papers Practiced  
6252                               2.0  

Predykcja Performance Index:
54.71185391749008

Rzeczywista wartość Performance Index:
51.0
